In [1]:
"""
Objectives: Demonstrate Claude's ability to request and receive fresh information by using Tools

Use Claude to set a reminder for future dates
"""

# Identify changes in source files and reload
%load_ext autoreload
%autoreload 2

# Import API key and Anthropic API
import json
import inspect
from dotenv import load_dotenv
from anthropic import Anthropic
from claude_chat import ClaudeChat
from tool_use_functions import ClaudeTool

In [2]:
# Access the API key
load_dotenv()

# Access the Anthropic API
client = Anthropic()

# Specify the model Claude will use
model = "claude-sonnet-5"

"""
Access:
- Conversation history related to creating an appointment for a future date
- Functions to store user inputs
- Functions to store Claude's responses
- Functions to interact with Claude client
"""
claude_chat = ClaudeChat(model, client)

"""
Access:
- Appointment reminder tool functions
- Tool function schema generation function
"""
claude_tool = ClaudeTool(model, client)

In [3]:
"""
Prompt engineering rules to replace prefilling
Rules for creating an appointment reminder for a future date
"""
appointment_reminder_prompt_rules = """
Rules:
- Return only valid string
- Do not use markdown
- Do not include comments
- Do not include explanations
- After the plain text, write END_OF_COMMANDS
"""

claude_chat.stop_sequences.append("END_OF_COMMANDS")

In [4]:
# Tool Function Use 1.: Observe how Claude responds without using tool functions
tooless_prompt = f"""
I am making an appointment for next Thursday. Return a reminder for an appointment for next Thursday with the actual date

{appointment_reminder_prompt_rules}
"""

claude_chat.userInput(tooless_prompt)

claude_response = claude_chat.askClaude()

claude_chat.claudeResponse(claude_response)

In [5]:
# Observe that Claude responded with the wrong date. Use tool functions in order to assist Claude
print(claude_response)

Reminder: You have an appointment scheduled for Thursday, June 12.


In [6]:
"""
Tool Function Use 2.: Generate tool function schemas for tool functions
Tool function schemas tell Claude what arguments the tool function expects, and how to use them
"""

# Give Claude the function tool syntax to generate the tool function schema
get_current_date_syntax = inspect.getsource(claude_tool.getCurrentDateTime)

current_date_schema = claude_tool.generateToolSchema(get_current_date_syntax)

In [7]:
# Observe the Claude generated tool schema for getCurrentDateTime()
parsed_current_date_schema = json.loads(current_date_schema)

print(json.dumps(parsed_current_date_schema, indent=2))

{
  "name": "getCurrentDateTime",
  "description": "Returns the current date and time formatted according to the specified format string",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "The strftime format string used to format the current date and time",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": [],
    "additionalProperties": false
  }
}


In [8]:
# Give Claude the function tool syntax to generate the tool function schema
calculate_future_date = inspect.getsource(claude_tool.calculateFutureDate)

future_date_schema = claude_tool.generateToolSchema(calculate_future_date)

In [9]:
# Observe the Claude generated tool schema for calculateFutureDate()
parsed_future_date_schema = json.loads(future_date_schema)

print(json.dumps(parsed_future_date_schema, indent=2))

{
  "name": "calculateFutureDate",
  "description": "Calculates the next occurrence of a specified weekday from the current date and returns it as a formatted date string",
  "input_schema": {
    "type": "object",
    "properties": {
      "weekday": {
        "type": "string",
        "description": "The name of the weekday to calculate the future date for (e.g., 'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday'). Case-insensitive and leading/trailing whitespace is ignored.",
        "enum": [
          "monday",
          "tuesday",
          "wednesday",
          "thursday",
          "friday",
          "saturday",
          "sunday"
        ]
      }
    },
    "required": [
      "weekday"
    ],
    "additionalProperties": false
  }
}


In [10]:
# Tool Function Use 3.: Make tool-enabled API calls
tool_enabled_current_date_prompt = f""""
What is today's date, formatted as Month, Date, Year

Here is an example input with an ideal response
<sample_input>
What is today's date, formatted as Month, Date, Year
</sample_input>

<ideal_output>
Today is July 21, 2026
</ideal_output>

{appointment_reminder_prompt_rules}
"""

claude_chat.userInput(tool_enabled_current_date_prompt)

"""
Return instructions about which tool to call and what parameters to use
- ID: Track the tool call
- Function name: Name of the function to call
- Input parameters: Formatted as a dictionary
"""
get_current_date_tool_request = claude_chat.askClaudeWithTools(parsed_current_date_schema)

claude_chat.claudeResponse(get_current_date_tool_request)

In [11]:
# Observe Claude's request to use the tool function
print(get_current_date_tool_request[1])

ToolUseBlock(id='toolu_01HPZT8dkbnbPEqxRpP2jeyT', caller=DirectCaller(type='direct'), input={'date_format': '%B %d, %Y'}, name='getCurrentDateTime', type='tool_use')


In [12]:
# Store the ToolUse id
get_current_date_id = get_current_date_tool_request[1].id

# Store the ToolUse input
get_current_date_input = get_current_date_tool_request[1].input

# Store the ToolUse function name
get_current_date_function_name = get_current_date_tool_request[1].name

In [13]:
# Tool Function Use 4.: Call the tool function and get the accurate answer from Claude

# Call the tool function using inputs from Claude
current_date = claude_tool.getCurrentDateTime(**get_current_date_input)

# Return the tool function results back to Claude
claude_chat.returnToolFunctionResult(current_date, get_current_date_id)

# Make the final API call
claude_current_date = claude_chat.askClaudeWithTools(parsed_current_date_schema )

In [14]:
# Observe Claude accurately identified today's date with tool enabled API calls
print(claude_current_date[0].text)

Today is July 22, 2026


In [15]:
# Tool Function Use 5.: Make the final tool enabled API call
tooless_prompt = f"""
I am making an appointment for next Thursday. Return a reminder for an appointment for next Thursday with the actual date

{appointment_reminder_prompt_rules}
"""

claude_chat.userInput(tooless_prompt)

claude_tooless_reminder = claude_chat.askClaude()

claude_chat.claudeResponse(claude_tooless_reminder)

In [16]:
# Observe that now since Claude knows today's date, the appointment reminder is more accurate
print(claude_tooless_reminder)

In [17]:
# Tool Function Use 6.: Make tool-enabled API call to do datetime math. This ensures Claude always returns the right future date
tool_enabled_future_date_prompt = f"""
I am making an appointment for next Thursday. Return a reminder for an appointment for next Thursday with the actual date

Here is an example input with an ideal response
<sample_input>
I am making an appointment in 2 weeks on Friday. Return a reminder for an appointment for next Thursday with the actual date
</sample_input>

<ideal_output>
You have an appointment reminder for: August 7, 2026
</ideal_output>

{appointment_reminder_prompt_rules}
"""

claude_chat.userInput(tool_enabled_future_date_prompt)

"""
Return instructions about which tool to call and what parameters to use
- ID: Track the tool call
- Function name: Name of the function to call
- Input parameters: Formatted as a dictionary
"""
get_future_date_tool_request = claude_chat.askClaudeWithTools(parsed_future_date_schema)

claude_chat.claudeResponse(get_future_date_tool_request)

In [18]:
print(get_future_date_tool_request)

[ToolUseBlock(id='toolu_017VcQwDwekdhiYVPg2grw48', caller=DirectCaller(type='direct'), input={'weekday': 'thursday'}, name='calculateFutureDate', type='tool_use')]


In [20]:
# Store the ToolUse id
get_future_date_id = get_future_date_tool_request[0].id

# Store the ToolUse input
get_future_date_input = get_future_date_tool_request[0].input

# Store the ToolUse function name
get_future_date_function_name = get_future_date_tool_request[0].name

In [23]:
# Tool Function Use 7.: Call the tool function and get the accurate answer from Claude

# Call the tool function using inputs from Claude
future_date = claude_tool.calculateFutureDate(**get_future_date_input)

# Return the tool function results back to Claude
claude_chat.returnToolFunctionResult(future_date, get_future_date_id)

# Make the final API call
claude_future_date = claude_chat.askClaudeWithTools(parsed_future_date_schema )

In [26]:
# Observe the appointment reminder now accurately displays the appointment date
print(claude_future_date[0].text)

You have an appointment reminder for: July 23, 2026
